In [2]:
import os
import mysql.connector
from google import genai
from dotenv import load_dotenv
import pandas as pd
from google.genai import types
import warnings
from IPython.display import display

# Desactivamos el aviso rojo de Pandas
warnings.filterwarnings('ignore', category=UserWarning)

# Cargamos variables de entorno
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
client = genai.Client(api_key=api_key)

def ejecutar_consulta_ia(pregunta_usuario):
    print(f"🤔 Procesando pregunta: '{pregunta_usuario}'...\n")
    
    # 1. Esquema limpio y preciso de tu base de datos
    esquema_base_datos = """
    Tablas MySQL disponibles:
    - GENERO (id_genero, nombre, descripcion)
    - AUTOR (id_autor, nombre, apellido, nacionalidad)
    - LIBRO (isbn, titulo, anio_publicacion, stock_total, stock_disponible)
    - SOCIO (id_socio, dni, nombre, apellido, email, fecha_alta, estado)
    - LIBRO_AUTOR (isbn, id_autor)
    - LIBRO_GENERO (isbn, id_genero)
    - EJEMPLAR (id_ejemplar, isbn, nro_ejemplar, estado_fisico)
    - SANCION (id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)
    - PRESTAMO (id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado)

    Vistas disponibles (usar preferentemente si aplica a la pregunta):
    - vw_prestamos_activos (id_prestamo, nombre_socio, email_socio, titulo_libro, fecha_vencimiento, dias_de_mora)
    - vw_libros_disponibles (isbn, titulo, autor, stock_disponible)
    - vw_socios_sancionados (id_socio, nombre_socio, estado_actual, total_sanciones_historicas)
    """
    
    # 2. Instrucciones directas e inquebrantables
    system_instruction = """
    Sos un traductor de lenguaje natural a SQL para MySQL. Tu única salida debe ser código SQL ejecutable.
    Cero explicaciones, cero saludos, cero formato Markdown (no uses ```sql ni ```).
    
    Reglas relacionales estrictas:
    1. LIBRO y AUTOR se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_AUTOR.
    2. LIBRO y GENERO se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_GENERO.
    3. En AUTOR y SOCIO, para buscar por nombre y apellido a la vez, usa: CONCAT(nombre, ' ', apellido).
    """
    
    # 3. Llamada a la IA y ejecución
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"{esquema_base_datos}\nPregunta del usuario: {pregunta_usuario}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        
        sql_query = response.text.strip()
        
        # Limpieza de seguridad por si manda bloques markdown
        if sql_query.startswith("```sql"): 
            sql_query = sql_query[6:]
        if sql_query.startswith("```"): 
            sql_query = sql_query[3:]
        if sql_query.endswith("```"): 
            sql_query = sql_query[:-3]
            
        sql_query = sql_query.strip()
            
        print(f"🤖 SQL generado por la IA:\n{sql_query}\n")
        
        # Conexión a la BD
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        if conexion.is_connected():
            df = pd.read_sql(sql_query, con=conexion)
            return df
            
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")
        return None
        
    finally:
        if 'conexion' in locals() and conexion.is_connected():
            conexion.close()
            

In [ ]:
#Pregunta1
pregunta =   "¿Cuáles son los 5 libros más prestados este año?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cuáles son los 5 libros más prestados este año?'...

🤖 SQL generado por la IA:
SELECT L.titulo, COUNT(P.id_prestamo) AS total_prestamos
FROM PRESTAMO AS P
JOIN EJEMPLAR AS E ON P.id_ejemplar = E.id_ejemplar
JOIN LIBRO AS L ON E.isbn = L.isbn
WHERE YEAR(P.fecha_prestamo) = YEAR(CURDATE())
GROUP BY L.isbn, L.titulo
ORDER BY total_prestamos DESC
LIMIT 5

📊 Resultados obtenidos:


,titulo,total_prestamos
0,Fundación,3
1,"Yo, Robot",3
2,El Resplandor,3
3,It (Eso),3
4,El Hobbit,3


In [ ]:
#Pregunta2
pregunta =  "¿Qué socios tienen préstamos vencidos en este momento?"  

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Qué socios tienen préstamos vencidos en este momento?'...

🤖 SQL generado por la IA:
SELECT DISTINCT nombre_socio, email_socio FROM vw_prestamos_activos WHERE dias_de_mora > 0;

📊 Resultados obtenidos:


,nombre_socio,email_socio
0,Juan Pérez,juan@mail.com
1,María Gómez,maria@mail.com
2,Carlos López,carlos@mail.com
3,Ana Martínez,ana@mail.com
4,Pedro Rodríguez,pedro@mail.com
5,Lucía Fernández,lucia@mail.com
6,Jorge García,jorge@mail.com
7,Sofía Díaz,sofia@mail.com
8,Luis Romero,luis@mail.com
9,David Suárez,david@mail.com


In [ ]:
#Pregunta3
pregunta =  "¿Cuántos ejemplares disponibles hay del libro con ISBN X?"  

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cuántos ejemplares disponibles hay del libro con ISBN X?'...

🤖 SQL generado por la IA:
SELECT stock_disponible FROM LIBRO WHERE isbn = 'X'

📭 No se encontraron resultados o la consulta falló.


In [ ]:
#Pregunta4
pregunta =  "¿Qué libros de ciencia ficción están disponibles para prestar?"  

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Qué libros de ciencia ficción están disponibles para prestar?'...

🤖 SQL generado por la IA:
SELECT
    L.titulo
FROM
    LIBRO AS L
JOIN
    LIBRO_GENERO AS LG ON L.isbn = LG.isbn
JOIN
    GENERO AS G ON LG.id_genero = G.id_genero
WHERE
    G.nombre = 'Ciencia Ficción' AND L.stock_disponible > 0;

📭 No se encontraron resultados o la consulta falló.


In [3]:
#Pregunta5
pregunta = "¿Cuál es el historial de préstamos del socio con DNI Y?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cuál es el historial de préstamos del socio con DNI Y?'...

🤖 SQL generado por la IA:
SELECT
    P.id_prestamo,
    P.fecha_prestamo,
    P.fecha_vencimiento,
    P.fecha_devolucion,
    P.estado,
    L.titulo AS titulo_libro,
    E.nro_ejemplar
FROM
    PRESTAMO AS P
JOIN
    SOCIO AS S ON P.id_socio = S.id_socio
JOIN
    EJEMPLAR AS E ON P.id_ejemplar = E.id_ejemplar
JOIN
    LIBRO AS L ON E.isbn = L.isbn
WHERE
    S.dni = 'Y';

📭 No se encontraron resultados o la consulta falló.


In [4]:
#Pregunta6
pregunta = "¿Qué autores tienen más de 3 libros en la biblioteca?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Qué autores tienen más de 3 libros en la biblioteca?'...

🤖 SQL generado por la IA:
SELECT A.nombre, A.apellido
FROM AUTOR AS A
JOIN LIBRO_AUTOR AS LA ON A.id_autor = LA.id_autor
GROUP BY A.id_autor, A.nombre, A.apellido
HAVING COUNT(LA.isbn) > 3

📭 No se encontraron resultados o la consulta falló.


In [5]:
#Pregunta7
pregunta = "¿Cuántas sanciones se generaron en el último mes?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cuántas sanciones se generaron en el último mes?'...

🤖 SQL generado por la IA:
SELECT COUNT(*) FROM SANCION WHERE fecha_inicio >= DATE_SUB(CURDATE(), INTERVAL 1 MONTH)

📊 Resultados obtenidos:


,COUNT(*)
0,2


In [6]:
#Pregunta8
pregunta = "¿cuantos socios poseen el mismo apellido?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿cuantos socios poseen el mismo apellido?'...

🤖 SQL generado por la IA:
SELECT apellido, COUNT(id_socio) AS total_socios
FROM SOCIO
GROUP BY apellido
HAVING COUNT(id_socio) > 1;

📭 No se encontraron resultados o la consulta falló.


In [7]:
#Pregunta9
pregunta = "¿Cual es la nacionalidad con mas libros publicados?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cual es la nacionalidad con mas libros publicados?'...

🤖 SQL generado por la IA:
SELECT A.nacionalidad
FROM AUTOR A
JOIN LIBRO_AUTOR LA ON A.id_autor = LA.id_autor
GROUP BY A.nacionalidad
ORDER BY COUNT(LA.isbn) DESC
LIMIT 1

📊 Resultados obtenidos:


,nacionalidad
0,Estadounidense


In [8]:
#Pregunta10
pregunta = "¿Cual es el libro mas popular (con mas prestamos)?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: '¿Cual es el libro mas popular (con mas prestamos)?'...

🤖 SQL generado por la IA:
SELECT L.titulo
FROM PRESTAMO P
JOIN EJEMPLAR E ON P.id_ejemplar = E.id_ejemplar
JOIN LIBRO L ON E.isbn = L.isbn
GROUP BY L.isbn, L.titulo
ORDER BY COUNT(P.id_prestamo) DESC
LIMIT 1

📊 Resultados obtenidos:


,titulo
0,Fundación
